In [13]:
import csv
import sys
import json
import pickle
import random
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
import kagglehub
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/My Drive/Kelompok9')
LOCAL_DIR = Path('/content')
DATASET_NAME = 'adityajn105/flickr8k'
ZIP_PATH = DRIVE_DIR / 'data' / 'flickr.zip'
EXTRACT_DIR = LOCAL_DIR / 'flickr8k'
IMAGES_DIR = EXTRACT_DIR / 'Images'
CAPTIONS_FILE = EXTRACT_DIR / 'captions.txt'
FEATURES_DIR = DRIVE_DIR / 'features'
RESULTS_DIR = DRIVE_DIR / 'results'
CAPTIONS_JSON = FEATURES_DIR / 'captions.json'
SPLITS_JSON = FEATURES_DIR / 'splits.json'
VOCAB_PKL = FEATURES_DIR / 'vocab.pkl'
STATS_JSON = RESULTS_DIR / 'dataset_stats.json'
HISTOGRAM_PNG = RESULTS_DIR / 'caption_length_histogram.png'
FEATURES_PT = FEATURES_DIR / 'features.pt'
SRC_DIR = DRIVE_DIR / 'src'

RANDOM_SEED = 42
VAL_SIZE = 1000
TEST_SIZE = 1000
FREQ_THRESHOLD = 5
BATCH_SIZE = 64

PAD_TOKEN = '<pad>'
START_TOKEN = '<start>'
END_TOKEN = '<end>'
UNK_TOKEN = '<unk>'

PAD_IDX = 0
START_IDX = 1
END_IDX = 2
UNK_IDX = 3

sys.path.append(str(SRC_DIR))
from dataset import get_loaders, get_eval_set, encode_caption

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Pemuatan Dataset

In [8]:
def download_dataset(dataset_name):
    return Path(kagglehub.dataset_download(dataset_name))

def archive_to_zip(source_dir, target_zip_path):
    target_zip_path.parent.mkdir(parents=True, exist_ok=True)
    base_name = str(target_zip_path.with_suffix(''))
    shutil.make_archive(base_name, 'zip', source_dir)

def extract_zip(source_zip_path, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source_zip_path, 'r') as archive:
        archive.extractall(target_dir)

downloaded_path = download_dataset(DATASET_NAME)
archive_to_zip(downloaded_path, ZIP_PATH)
extract_zip(ZIP_PATH, EXTRACT_DIR)

Using Colab cache for faster access to the 'flickr8k' dataset.


## Pembersihan Captions

In [9]:
def load_raw_captions(file_path):
    raw_dict = {}
    with open(file_path, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            image_name = row[0]
            caption = row[1]
            if image_name not in raw_dict:
                raw_dict[image_name] = []
            raw_dict[image_name].append(caption)
    return raw_dict

def clean_caption(caption):
    caption = caption.lower()
    caption = re.sub(r"['’]", "", caption)
    caption = re.sub(r"[^a-z\s]", " ", caption)
    return " ".join(caption.split())

def clean_caption_dict(raw_dict):
    cleaned_dict = {}
    for image_name, captions in raw_dict.items():
        cleaned_dict[image_name] = [clean_caption(c) for c in captions]
    return cleaned_dict

def save_json(data, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=2)

raw_captions = load_raw_captions(CAPTIONS_FILE)
cleaned_captions = clean_caption_dict(raw_captions)
save_json(cleaned_captions, CAPTIONS_JSON)

## Data Splitting

In [10]:
def load_captions(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def split_image_ids(image_ids, val_size, test_size, seed):
    shuffled_ids = sorted(image_ids)
    rng = random.Random(seed)
    rng.shuffle(shuffled_ids)
    train_size = len(shuffled_ids) - val_size - test_size
    train_ids = shuffled_ids[:train_size]
    val_ids = shuffled_ids[train_size:train_size + val_size]
    test_ids = shuffled_ids[train_size + val_size:]
    return {
        "train": train_ids,
        "val": val_ids,
        "test": test_ids
    }

def count_captions(captions_dict, image_ids):
    return sum(len(captions_dict[img_id]) for img_id in image_ids)

def display_split_summary(splits, captions_dict):
    for split_name, ids in splits.items():
        total_images = len(ids)
        total_captions = count_captions(captions_dict, ids)
        print(f"{split_name.capitalize()} -> Images: {total_images}, Captions: {total_captions}")

def save_splits(splits_dict, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as file:
        json.dump(splits_dict, file, indent=2)

captions_data = load_captions(CAPTIONS_JSON)
dataset_splits = split_image_ids(list(captions_data.keys()), VAL_SIZE, TEST_SIZE, RANDOM_SEED)
display_split_summary(dataset_splits, captions_data)
save_splits(dataset_splits, SPLITS_JSON)

Train -> Images: 6091, Captions: 30455
Val -> Images: 1000, Captions: 5000
Test -> Images: 1000, Captions: 5000


## Pembuatan Vocabulary

In [11]:
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def get_training_tokens(captions_dict, train_image_ids):
    tokens = []
    for img_id in train_image_ids:
        for caption in captions_dict[img_id]:
            tokens.extend(caption.split())
    return tokens

def build_vocabulary(tokens, min_freq):
    word_counts = Counter(tokens)
    filtered_words = [word for word, count in word_counts.items() if count >= min_freq]
    filtered_words.sort(key=lambda word: (-word_counts[word], word))
    word2idx = {
        PAD_TOKEN: PAD_IDX,
        START_TOKEN: START_IDX,
        END_TOKEN: END_IDX,
        UNK_TOKEN: UNK_IDX
    }
    for word in filtered_words:
        if word not in word2idx:
            word2idx[word] = len(word2idx)
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

def get_caption_lengths(captions_dict):
    lengths = []
    for captions in captions_dict.values():
        for caption in captions:
            lengths.append(len(caption.split()))
    return lengths

def compute_mean(values):
    return sum(values) / len(values)

def create_dataset_stats(vocab_size, lengths):
    return {
        "vocab_size": vocab_size,
        "total_captions": len(lengths),
        "avg_caption_length": round(compute_mean(lengths), 2),
        "min_caption_length": min(lengths),
        "max_caption_length": max(lengths)
    }

def save_histogram(lengths, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8, 5))
    plt.hist(lengths, bins=range(1, max(lengths) + 2), edgecolor='black', align='left')
    plt.xlabel('Caption Length (Words)')
    plt.ylabel('Frequency')
    plt.title('Caption Length Distribution')
    plt.savefig(output_path, bbox_inches='tight')
    plt.close()

def save_pickle(data, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'wb') as file:
        pickle.dump(data, file)

def save_json(data, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=2)

captions_data = load_json(CAPTIONS_JSON)
splits_data = load_json(SPLITS_JSON)

train_tokens = get_training_tokens(captions_data, splits_data['train'])
word2idx, idx2word = build_vocabulary(train_tokens, FREQ_THRESHOLD)

vocab_payload = {
    'word2idx': word2idx,
    'idx2word': idx2word
}
save_pickle(vocab_payload, VOCAB_PKL)

caption_lengths = get_caption_lengths(captions_data)
dataset_stats = create_dataset_stats(len(word2idx), caption_lengths)
save_json(dataset_stats, STATS_JSON)
save_histogram(caption_lengths, HISTOGRAM_PNG)

print(dataset_stats)
sample_img = splits_data['train'][0]
sample_cap = captions_data[sample_img][0]
encoded_sample = encode_caption(sample_cap, word2idx)
print(f"Contoh ID Token (Length: {len(encoded_sample)}): {encoded_sample}")

{'vocab_size': 2584, 'total_captions': 40455, 'avg_caption_length': 10.81, 'min_caption_length': 1, 'max_caption_length': 36}
Contoh ID Token (Length: 30): [1, 4, 16, 9, 15, 10, 8, 35, 11, 4, 715, 5, 4, 54, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Ekstraksi Fitur

In [14]:
class ImageExtractionDataset(Dataset):
    def __init__(self, image_names, images_dir, transform):
        self.image_names = image_names
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = self.images_dir / img_name
        image = Image.open(img_path).convert('RGB')
        return img_name, self.transform(image)

def get_resnet_transform():
    weights = ResNet50_Weights.IMAGENET1K_V2
    return weights.transforms()

def print_transform_parameters(transform):
    print("Parameter Transformasi Resmi:")
    print(f"Resize Size: {transform.resize_size}")
    print(f"Crop Size: {transform.crop_size}")
    print(f"Interpolation: {transform.interpolation}")
    print(f"Mean: {transform.mean}")
    print(f"Std: {transform.std}")

def build_feature_extractor(device):
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights)
    model.fc = nn.Identity()
    for param in model.parameters():
        param.requires_grad = False
    model.eval()
    return model.to(device)

def extract_features(model, dataloader, device):
    features_dict = {}
    with torch.no_grad():
        for names, images in dataloader:
            images = images.to(device)
            outputs = model(images)
            outputs = outputs.cpu()
            for name, feat in zip(names, outputs):
                features_dict[name] = feat
    return features_dict

def save_features(features_dict, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(features_dict, output_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = get_resnet_transform()
print_transform_parameters(transform)

captions_data = load_json(CAPTIONS_JSON)
all_image_names = list(captions_data.keys())

dataset = ImageExtractionDataset(all_image_names, IMAGES_DIR, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

extractor_model = build_feature_extractor(device)
extracted_features = extract_features(extractor_model, dataloader, device)
save_features(extracted_features, FEATURES_PT)

print(f"Total Fitur Gambar Tersimpan: {len(extracted_features)}")
sample_img_id = all_image_names[0]
print(f"Contoh Tensor Fitur ({sample_img_id}) Shape: {extracted_features[sample_img_id].shape}, Dtype: {extracted_features[sample_img_id].dtype}")

Parameter Transformasi Resmi:
Resize Size: [232]
Crop Size: [224]
Interpolation: InterpolationMode.BILINEAR
Mean: [0.485, 0.456, 0.406]
Std: [0.229, 0.224, 0.225]
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s]


Total Fitur Gambar Tersimpan: 8091
Contoh Tensor Fitur (1000268201_693b08cb0e.jpg) Shape: torch.Size([2048]), Dtype: torch.float32


## Uji Coba Data Loader

In [15]:
train_loader, val_loader, test_loader, vocab = get_loaders(batch_size=64)
batch_features, batch_captions = next(iter(train_loader))

print(f"Train Dataset Length: {len(train_loader.dataset)}")
print(f"Val Dataset Length: {len(val_loader.dataset)}")
print(f"Test Dataset Length: {len(test_loader.dataset)}")
print(f"Batch Features Shape: {batch_features.shape}, Dtype: {batch_features.dtype}")
print(f"Batch Captions Shape: {batch_captions.shape}, Dtype: {batch_captions.dtype}")

val_names, val_feats, val_refs = get_eval_set('val')
print(f"Eval Names Count: {len(val_names)}")
print(f"Eval Features Shape: {val_feats.shape}, Dtype: {val_feats.dtype}")
print(f"Eval References Count: {len(val_refs)}")
print(f"Contoh 5 Referensi Pertama Gambar 1: {val_refs[0]}")

Train Dataset Length: 30455
Val Dataset Length: 5000
Test Dataset Length: 5000
Batch Features Shape: torch.Size([64, 2048]), Dtype: torch.float32
Batch Captions Shape: torch.Size([64, 30]), Dtype: torch.int64
Eval Names Count: 1000
Eval Features Shape: torch.Size([1000, 2048]), Dtype: torch.float32
Eval References Count: 1000
Contoh 5 Referensi Pertama Gambar 1: [['a', 'young', 'boy', 'wearing', 'a', 'blue', 'shirt', 'is', 'playing', 'at', 'a', 'park'], ['children', 'playing', 'on', 'a', 'playground'], ['the', 'boy', 'and', 'girl', 'are', 'playing', 'on', 'the', 'outdoor', 'gym'], ['the', 'boy', 'is', 'wearing', 'a', 'blue', 'shirt', 'and', 'the', 'girl', 'is', 'wearing', 'black', 'clothes'], ['the', 'two', 'kids', 'are', 'playing', 'at', 'the', 'playground']]
